# Model Validation Diagnostics

This notebook validates the final latent potential outputs for Smil Labs. It checks submission shape, uplift behavior, segment summaries, POI/catchment relationships, and top-risk predictions.

In [1]:
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / 'Results'
GOLD_DIR = PROJECT_ROOT / 'data' / 'gold'
DOCS_DIR = PROJECT_ROOT / 'Docs'

PLATFORM_FILE = RESULTS_DIR / 'smil_labs_predictions.csv'
FULL_FILE = RESULTS_DIR / 'smil_labs_predictions_full_20000.csv'
DIAGNOSTICS_FILE = GOLD_DIR / 'prediction_diagnostics.csv'

platform = pd.read_csv(PLATFORM_FILE)
full = pd.read_csv(FULL_FILE)
diagnostics = pd.read_csv(DIAGNOSTICS_FILE)

print(platform.shape, full.shape, diagnostics.shape)

(914, 2) (20000, 2) (20000, 16)


## Schema Checks

In [2]:
schema_checks = pd.DataFrame([
    {'check': 'platform_rows', 'value': len(platform), 'expected': 914, 'passed': len(platform) == 914},
    {'check': 'platform_columns', 'value': ', '.join(platform.columns), 'expected': 'row_id, Maximum_Monthly_Liters', 'passed': list(platform.columns) == ['row_id', 'Maximum_Monthly_Liters']},
    {'check': 'platform_missing_values', 'value': int(platform.isna().sum().sum()), 'expected': 0, 'passed': int(platform.isna().sum().sum()) == 0},
    {'check': 'platform_unique_row_id', 'value': platform['row_id'].nunique(), 'expected': len(platform), 'passed': platform['row_id'].nunique() == len(platform)},
    {'check': 'full_rows', 'value': len(full), 'expected': 20000, 'passed': len(full) == 20000},
    {'check': 'full_missing_values', 'value': int(full.isna().sum().sum()), 'expected': 0, 'passed': int(full.isna().sum().sum()) == 0},
])
schema_checks

,check,value,expected,passed
0,platform_rows,914,914,True
1,platform_columns,"row_id, Maximum_Monthly_Liters","row_id, Maximum_Monthly_Liters",True
2,platform_missing_values,0,0,True
3,platform_unique_row_id,914,914,True
4,full_rows,20000,20000,True
5,full_missing_values,0,0,True


## Prediction and Uplift Distribution

In [3]:
distribution = diagnostics[
    ['observed_max_monthly_liters', 'Maximum_Monthly_Liters', 'uplift_ratio_vs_max', 'constraint_score', 'catchment_density_score', 'poi_demand_score']
].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]).round(3)
distribution

,observed_max_monthly_liters,Maximum_Monthly_Liters,uplift_ratio_vs_max,constraint_score,catchment_density_score,poi_demand_score
count,20000.000,20000.000,20000.000,20000.000,20000.000,20000.0
mean,396.059,445.025,1.357,0.166,0.500,0.0
std,480.360,464.411,0.452,0.074,0.263,0.0
min,28.051,84.153,0.994,0.000,0.012,0.0
1%,75.437,124.749,0.999,0.013,0.012,0.0
5%,88.348,138.954,1.002,0.042,0.101,0.0
10%,96.072,148.826,1.008,0.065,0.153,0.0
25%,113.663,171.315,1.060,0.115,0.272,0.0
50%,164.047,258.498,1.181,0.170,0.463,0.0
75%,346.141,388.698,1.454,0.216,0.750,0.0


## Segment Validation

In [4]:
by_size = diagnostics.groupby('Outlet_Size').agg(
    outlets=('Outlet_ID', 'count'),
    avg_potential=('Maximum_Monthly_Liters', 'mean'),
    median_potential=('Maximum_Monthly_Liters', 'median'),
    avg_uplift=('uplift_ratio_vs_max', 'mean'),
    avg_constraint=('constraint_score', 'mean'),
    avg_poi_score=('poi_demand_score', 'mean'),
).round(3).sort_values('avg_potential', ascending=False)

by_type = diagnostics.groupby('Outlet_Type').agg(
    outlets=('Outlet_ID', 'count'),
    avg_potential=('Maximum_Monthly_Liters', 'mean'),
    median_potential=('Maximum_Monthly_Liters', 'median'),
    avg_uplift=('uplift_ratio_vs_max', 'mean'),
    avg_constraint=('constraint_score', 'mean'),
    avg_poi_score=('poi_demand_score', 'mean'),
).round(3).sort_values('avg_potential', ascending=False)

by_distributor = diagnostics.groupby('Distributor_ID').agg(
    outlets=('Outlet_ID', 'count'),
    avg_potential=('Maximum_Monthly_Liters', 'mean'),
    avg_uplift=('uplift_ratio_vs_max', 'mean'),
    avg_constraint=('constraint_score', 'mean'),
).round(3).sort_values('avg_potential', ascending=False)

display(by_size)
display(by_type)
display(by_distributor)

,outlets,avg_potential,median_potential,avg_uplift,avg_constraint,avg_poi_score
Outlet_Size,,,,,,
Extra Large,943,2064.067,2050.767,1.014,0.242,0.0
Large,2887,968.583,961.164,1.011,0.219,0.0
Medium,5702,369.455,357.245,1.163,0.161,0.0
Small,10272,195.971,173.734,1.590,0.148,0.0
Unknown,196,194.624,168.896,1.515,0.160,0.0


,outlets,avg_potential,median_potential,avg_uplift,avg_constraint,avg_poi_score
Outlet_Type,,,,,,
Hotel,2797,455.073,251.758,1.411,0.180,0.0
Grocery,3158,450.600,262.318,1.367,0.167,0.0
Pharmacy,2691,448.625,230.078,1.287,0.151,0.0
Eatery,2867,447.098,272.531,1.388,0.173,0.0
Kiosk,2691,442.006,238.405,1.274,0.148,0.0
SMMT,2723,437.253,262.409,1.445,0.187,0.0
Bakery,3073,434.596,237.948,1.325,0.157,0.0


,outlets,avg_potential,avg_uplift,avg_constraint
Distributor_ID,,,,
DIST_W_03,2991,465.511,1.451,0.186
DIST_W_01,3020,462.310,1.460,0.188
DIST_W_02,2989,458.002,1.429,0.185
DIST_C_03,1296,447.730,1.283,0.152
DIST_C_01,1385,440.726,1.273,0.152
DIST_NW_01,2015,436.991,1.294,0.154
DIST_NW_02,1985,430.856,1.296,0.154
DIST_C_02,1319,429.911,1.275,0.151
DIST_S_02,1495,426.211,1.278,0.143


## Top Potential and Top Uplift Reviews

In [5]:
review_columns = [
    'Outlet_ID', 'Outlet_Size', 'Outlet_Type', 'Distributor_ID',
    'observed_max_monthly_liters', 'Maximum_Monthly_Liters', 'uplift_ratio_vs_max',
    'constraint_score', 'catchment_density_score', 'poi_demand_score',
]

top_potential = diagnostics.sort_values('Maximum_Monthly_Liters', ascending=False).head(100)[review_columns]
top_uplift = diagnostics.sort_values('uplift_ratio_vs_max', ascending=False).head(100)[review_columns]

top_potential.to_csv(GOLD_DIR / 'validation_top_100_potential.csv', index=False)
top_uplift.to_csv(GOLD_DIR / 'validation_top_100_uplift.csv', index=False)

display(top_potential.head(10).round(3))
display(top_uplift.head(10).round(3))

,Outlet_ID,Outlet_Size,Outlet_Type,Distributor_ID,observed_max_monthly_liters,Maximum_Monthly_Liters,uplift_ratio_vs_max,constraint_score,catchment_density_score,poi_demand_score
18994,OUT_18995,Extra Large,Grocery,DIST_S_02,10457.941,10457.941,1.0,0.010,0.128,0.0
2961,OUT_02962,Extra Large,Eatery,DIST_W_03,3004.816,3004.816,1.0,0.229,0.753,0.0
10022,OUT_10023,Extra Large,Hotel,DIST_C_02,2994.568,2994.568,1.0,0.237,0.355,0.0
12986,OUT_12987,Extra Large,Pharmacy,DIST_C_02,2986.193,2986.193,1.0,0.242,0.466,0.0
5354,OUT_05355,Extra Large,Grocery,DIST_W_03,2912.573,2912.573,1.0,0.227,0.755,0.0
9500,OUT_09501,Extra Large,Grocery,DIST_C_02,2895.887,2895.887,1.0,0.241,0.458,0.0
18984,OUT_18985,Extra Large,Kiosk,DIST_S_02,2895.270,2895.270,1.0,0.228,0.450,0.0
12587,OUT_12588,Extra Large,Eatery,DIST_C_03,2884.929,2884.929,1.0,0.239,0.475,0.0
13752,OUT_13753,Extra Large,SMMT,DIST_NW_02,2884.642,2884.642,1.0,0.241,0.086,0.0
9274,OUT_09275,Extra Large,Pharmacy,DIST_C_01,2881.758,2881.758,1.0,0.240,0.386,0.0


,Outlet_ID,Outlet_Size,Outlet_Type,Distributor_ID,observed_max_monthly_liters,Maximum_Monthly_Liters,uplift_ratio_vs_max,constraint_score,catchment_density_score,poi_demand_score
5658,OUT_05659,Small,Bakery,DIST_W_01,101.017,303.052,2.971,0.416,0.879,0.0
10202,OUT_10203,Small,Grocery,DIST_C_02,97.916,293.747,2.970,0.389,0.400,0.0
8086,OUT_08087,Small,Grocery,DIST_W_03,97.758,293.273,2.970,0.395,0.828,0.0
2158,OUT_02159,Small,Eatery,DIST_W_03,96.669,290.008,2.969,0.361,0.794,0.0
6138,OUT_06139,Small,Eatery,DIST_W_02,94.839,284.517,2.969,0.424,0.881,0.0
6540,OUT_06541,Small,Eatery,DIST_W_01,94.674,284.023,2.969,0.368,0.844,0.0
5668,OUT_05669,Small,Grocery,DIST_W_02,94.583,283.749,2.969,0.411,0.764,0.0
3752,OUT_03753,Small,Grocery,DIST_W_01,94.554,283.663,2.969,0.378,0.775,0.0
457,OUT_00458,Small,Grocery,DIST_W_02,94.425,283.276,2.969,0.376,0.920,0.0
2003,OUT_02004,Small,Eatery,DIST_W_01,94.001,282.002,2.968,0.370,0.885,0.0


## Signal Correlation

In [6]:
correlation = diagnostics[
    ['poi_demand_score', 'catchment_density_score', 'constraint_score', 'Maximum_Monthly_Liters', 'uplift_ratio_vs_max']
].corr().round(3)
correlation

,poi_demand_score,catchment_density_score,constraint_score,Maximum_Monthly_Liters,uplift_ratio_vs_max
poi_demand_score,NaN,NaN,NaN,NaN,NaN
catchment_density_score,NaN,1.000,0.248,0.030,0.182
constraint_score,NaN,0.248,1.000,0.408,0.521
Maximum_Monthly_Liters,NaN,0.030,0.408,1.000,-0.370
uplift_ratio_vs_max,NaN,0.182,0.521,-0.370,1.000


## Write Validation Summary

In [7]:
def md_table(df: pd.DataFrame) -> str:
    display_df = df.reset_index() if df.index.name is not None else df.copy()
    display_df = display_df.where(pd.notna(display_df), '')
    lines = ['| ' + ' | '.join(map(str, display_df.columns)) + ' |']
    lines.append('| ' + ' | '.join(['---'] * len(display_df.columns)) + ' |')
    for row in display_df.itertuples(index=False, name=None):
        lines.append('| ' + ' | '.join(map(str, row)) + ' |')
    return '\n'.join(lines)

lines = [
    '# Model Validation Summary',
    '',
    'This validation summary was generated from `Notebooks/04_model_validation.ipynb`.',
    '',
    '## Schema Checks',
    '',
    md_table(schema_checks),
    '',
    '## Key Distribution Metrics',
    '',
    md_table(distribution.loc[['mean', '50%', '90%', '95%', '99%', 'max']].reset_index()),
    '',
    '## Outlet Size Summary',
    '',
    md_table(by_size.reset_index()),
    '',
    '## Outlet Type Summary',
    '',
    md_table(by_type.reset_index()),
    '',
    '## Signal Correlation',
    '',
    md_table(correlation.reset_index()),
    '',
    '## Review Artifacts',
    '',
    '- `data/gold/validation_top_100_potential.csv`',
    '- `data/gold/validation_top_100_uplift.csv`',
]

(DOCS_DIR / 'model_validation_summary.md').write_text('\n'.join(lines), encoding='utf-8')
print(DOCS_DIR / 'model_validation_summary.md')

d:\projects\Data-Storm-2026\Docs\model_validation_summary.md
